# 08 • Datas, calendário e janelas

## Preparar esta sessão
Execute as células abaixo antes de começar a aula. Elas localizam os arquivos e preparam os objetos necessários. Você não precisa decorar essa preparação agora.

In [1]:
from pathlib import Path
import pandas as pd
import math
BASE = Path.cwd()
if not (BASE / 'dados').exists() and (BASE.parent / 'dados').exists():
    BASE = BASE.parent
DADOS = BASE / 'dados'
SAIDAS = BASE / 'resultados'
SAIDAS.mkdir(exist_ok=True)
assert (DADOS/'penguins.csv').exists(), 'Descompacte o curso inteiro e abra na pasta correta.'


## Preparação independente
Execute esta célula para recriar os objetos necessários dos módulos anteriores.

In [2]:
p = pd.read_csv(DADOS / 'penguins.csv')
print(p.head(3).to_string(index=False))
print('Dimensões:', p.shape)
print('Colunas:', p.columns.tolist())
print(type(p['species']).__name__)

mascara = p['species'].eq('Adelie') & p['body_mass_g'].gt(4000)
print(p.loc[mascara, ['species', 'island', 'body_mass_g']].head())
print(p.iloc[:3, :2])

qualidade = pd.DataFrame({'ausentes':p.isna().sum(), 'percentual':p.isna().mean().mul(100)})
print(qualidade)
medidos = p.dropna(subset=['body_mass_g']).copy()
print('Observações com massa:', len(medidos))

enriquecido = p.assign(body_mass_kg=p['body_mass_g']/1000)
print(enriquecido[['body_mass_g','body_mass_kg']].head())
print(p['species'].astype('string').str.lower().head())

resumo = p.groupby('species', as_index=False, observed=True).agg(
    observacoes=('body_mass_g','size'),
    massas_validas=('body_mass_g','count'),
    massa_media_g=('body_mass_g','mean'))
print(resumo)

nomes = pd.DataFrame({'species':['Adelie','Chinstrap','Gentoo'],
                      'nome_pt':['Adélia','Barbicha','Papua']})
cruzado = p.merge(nomes, on='species', how='left', validate='many_to_one', indicator=True)
print(cruzado['_merge'].value_counts())

r = pd.read_csv(DADOS/'online_retail.csv', dtype={
    'InvoiceNo':'string','StockCode':'string','CustomerID':'string'})
r['InvoiceDate'] = pd.to_datetime(r['InvoiceDate'], errors='coerce')
cancelada = r['InvoiceNo'].str.upper().str.startswith('C', na=False)
elegivel = (r['Quantity'].gt(0) & r['UnitPrice'].gt(0)
            & r['InvoiceDate'].notna() & ~cancelada)
v = r.loc[elegivel].copy()
v['valor'] = v['Quantity'] * v['UnitPrice']
print('Linhas brutas:',len(r),'Linhas elegíveis:',len(v))
print(r.isna().sum())

species    island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g    sex  year
 Adelie Torgersen            39.1           18.7              181.0       3750.0   male  2007
 Adelie Torgersen            39.5           17.4              186.0       3800.0 female  2007
 Adelie Torgersen            40.3           18.0              195.0       3250.0 female  2007
Dimensões: (344, 8)
Colunas: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
Series
   species     island  body_mass_g
7   Adelie  Torgersen       4675.0
9   Adelie  Torgersen       4250.0
14  Adelie  Torgersen       4400.0
17  Adelie  Torgersen       4500.0
19  Adelie  Torgersen       4200.0
  species     island
0  Adelie  Torgersen
1  Adelie  Torgersen
2  Adelie  Torgersen
                   ausentes  percentual
species                   0    0.000000
island                    0    0.000000
bill_length_mm            2    0.581395
bill_depth_mm             

### Uma data é mais que um texto

O texto '2026-01-31' pode ser exibido, mas operações de calendário precisam de um tipo de data. to_datetime faz essa conversão. Depois, `v['InvoiceDate'].dt.year` extrai o ano; `.dt.to_period('M')` identifica o ano-mês. Agrupar somente por nome ou número do mês mistura janeiro de anos diferentes; use ano e mês juntos.

No exemplo guiado, set_index coloca InvoiceDate como índice temporário para resample. O código não substitui v. resample('MS') cria grupos mensais rotulados pelo início de cada mês; sum soma valores dentro de cada grupo. min_count=1 evita que um grupo totalmente sem valores presentes seja apresentado automaticamente como zero.

Para entender uma janela móvel, imagine valores diários fictícios 10, 20 e 30. Uma janela de duas observações produz primeiro ausência, depois 15 e depois 25, quando exige duas observações. Ela desliza pelo calendário. Antes de interpretar “sete linhas” como “sete dias”, confirme que existe uma linha por dia. shift(1) desloca o valor anterior para a linha atual, permitindo calcular (atual/anterior − 1) × 100. O primeiro período não tem anterior e deve ficar sem variação calculada.

### Aprofundamento e cuidados

`to_datetime` converte datas; o acessor `dt` extrai mês, dia e hora. Datas ambíguas como 03/04/2026 precisam de um formato explícito, por exemplo `%d/%m/%Y`. Na base da loja, trabalhamos com o horário registrado, sem informação de fuso; não inventamos uma conversão para UTC.

`resample` agrupa por intervalos de tempo quando o índice é de datas. Usaremos `MS`, início do mês. `rolling(7)` usa sete observações; `rolling("7D")` usa uma janela de tempo. Para comparar dias, reindexe o calendário e declare se ausência significa zero vendas ou falha na coleta. Aqui preencheremos com zero dentro do intervalo observado, como hipótese do exercício.

O primeiro e o último mês podem estar incompletos. A base termina em dezembro de 2011 antes do fim do mês; comparar seu total com novembro pode sugerir uma queda artificial. Crescimento percentual não é definido quando o valor anterior é zero. Valores monetários em float são adequados para esta exploração, mas sistemas financeiros exigem política explícita de precisão e arredondamento.


## Exemplo guiado

In [3]:
mensal = v.set_index('InvoiceDate')['valor'].resample('MS').sum(min_count=1)
print(mensal)
print('Cobertura:',r['InvoiceDate'].min(),r['InvoiceDate'].max())

InvoiceDate
2010-12-01     823746.140
2011-01-01     691364.560
2011-02-01     523631.890
2011-03-01     717639.360
2011-04-01     537808.621
2011-05-01     770536.020
2011-06-01     761739.900
2011-07-01     719221.191
2011-08-01     759138.380
2011-09-01    1058590.172
2011-10-01    1154979.300
2011-11-01    1509496.330
2011-12-01     638792.680
Freq: MS, Name: valor, dtype: float64
Cobertura: 2010-12-01 08:26:00 2011-12-09 12:50:00


## Exercício 08.1
Some receita elegível por ano e mês usando dt.to_period.

In [4]:
print(v.groupby(v['InvoiceDate'].dt.to_period('M'))['valor'].sum())

InvoiceDate
2010-12     823746.140
2011-01     691364.560
2011-02     523631.890
2011-03     717639.360
2011-04     537808.621
2011-05     770536.020
2011-06     761739.900
2011-07     719221.191
2011-08     759138.380
2011-09    1058590.172
2011-10    1154979.300
2011-11    1509496.330
2011-12     638792.680
Freq: M, Name: valor, dtype: float64


## Exercício 08.2
Calcule a receita por dia e complete o calendário com zero, documentando a hipótese.

In [5]:
diario = v.groupby(v['InvoiceDate'].dt.normalize())['valor'].sum()
diario = diario.reindex(pd.date_range(diario.index.min(),diario.index.max(),freq='D'), fill_value=0)
print(diario.head())
# Hipótese: dias não registrados dentro da cobertura tiveram zero venda elegível.

2010-12-01    58960.79
2010-12-02    47748.38
2010-12-03    46943.71
2010-12-04        0.00
2010-12-05    31774.95
Freq: D, Name: valor, dtype: float64


## Exercício 08.3
Calcule a média móvel de sete dias, exigindo sete dias para produzir resultado.

In [6]:
print(diario.rolling(7,min_periods=7).mean().head(10))

2010-12-01             NaN
2010-12-02             NaN
2010-12-03             NaN
2010-12-04             NaN
2010-12-05             NaN
2010-12-06             NaN
2010-12-07    48553.784286
2010-12-08    46615.097143
2010-12-09    47449.068571
2010-12-10    49197.527143
Freq: D, Name: valor, dtype: float64


## Exercício 08.4
Calcule variação mensal e substitua resultados com denominador anterior zero por ausência.

In [7]:
anterior = mensal.shift(1)
crescimento = mensal.div(anterior.where(anterior.ne(0))).sub(1).mul(100)
print(crescimento)
# Avalie cobertura antes de interpretar dezembro.

InvoiceDate
2010-12-01          NaN
2011-01-01   -16.070677
2011-02-01   -24.261103
2011-03-01    37.050354
2011-04-01   -25.058650
2011-05-01    43.273274
2011-06-01    -1.141559
2011-07-01    -5.581788
2011-08-01     5.550057
2011-09-01    39.446272
2011-10-01     9.105424
2011-11-01    30.694665
2011-12-01   -57.681733
Freq: MS, Name: valor, dtype: float64


## Exercício 08.5
Converta ["31/01/2026", "data ruim"] com formato brasileiro e conte falhas.

In [8]:
datas = pd.to_datetime(pd.Series(['31/01/2026','data ruim']),format='%d/%m/%Y',errors='coerce')
print(datas); print('Falhas:',datas.isna().sum())

0   2026-01-31
1          NaT
dtype: datetime64[us]
Falhas: 1
